# PPO Fine-tuning — CPMP Transformer V9

Fine-tuning del modelo pre-entrenado usando **PPO (Proximal Policy Optimization)**.

### Mejoras sobre REINFORCE
- **Reward por paso**: señal densa en cada movimiento (`n_sorted_después - n_sorted_antes`)
- **GAE**: Generalized Advantage Estimation (γ=0.99, λ=0.95) reduce varianza de las ventajas
- **Value head (Critic)**: red aprendida que estima V(s) → baseline adaptativo por estado
- **PPO clip**: previene updates demasiado grandes → entrenamiento más estable
- **K epochs por rollout**: cada batch de experiencias se reutiliza 4 veces → más eficiente
- **batch_size=128**: 4× más instancias por rollout que REINFORCE (aprovecha la GPU)

### Archivos de salida
- Checkpoint: `models/v9_rl_ppo_checkpoint.pth` (no sobreescribe los resultados de REINFORCE)

## 1. Setup

In [1]:
import sys
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print('CWD:', os.getcwd())
print('src:', src_path)

CWD: /mnt/data/Proyectos/Universidad/CPMP-Transformer
src: /mnt/data/Proyectos/Universidad/CPMP-Transformer/src


In [2]:
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, ConcatDataset

from models.cpmp_transformer_v9 import CPMPTransformer
from training.training import load_model, load_hyperparams, pad_batch_collate
from training.ppo_training import PPOActorCritic, PPOTrainer
from generation.instances import generate_instances, read_instance
from preprocessing.dataset import H5Dataset
from settings import INSTANCE_FOLDER, DATA_FOLDER, MODELS_FOLDER

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Dispositivo: cuda
GPU: NVIDIA GeForce RTX 5060
VRAM: 7.5 GB


## 2. Generar instancias de entrenamiento PPO

Mismas 4 configuraciones que REINFORCE — reutiliza las instancias ya generadas si existen.

In [3]:
RL_CONFIGS = [
    (5,  4, 15, 'rl_E4',  5000),   # Pequeños
    (7,  5, 25, 'rl_E5',  5000),   # Intermedios
    (8,  6, 40, 'rl_E6b', 5000),   # Medianos-difíciles
    (6,  7, 30, 'rl_E7',  5000),   # Anchos
]

RL_SEED   = 100
R_DESORDEN = 50
MAX_STEPS  = 100

In [4]:
print('Generando instancias PPO...')
for H, S, N, basename, amount in RL_CONFIGS:
    folder = INSTANCE_FOLDER / basename
    if folder.exists() and len(list(folder.glob('*.txt'))) >= amount:
        print(f'  {basename}: ya existen {amount} instancias, saltando.')
        continue
    generate_instances(basename, H, S, N, amount, r=R_DESORDEN, seed=RL_SEED)
    print(f'  {basename}: {amount} instancias generadas.')
print('Listo.')

Generando instancias PPO...
  rl_E4: ya existen 5000 instancias, saltando.
  rl_E5: ya existen 5000 instancias, saltando.
  rl_E6b: ya existen 5000 instancias, saltando.
  rl_E7: ya existen 5000 instancias, saltando.
Listo.


In [5]:
ppo_layouts = []
for H, S, N, basename, amount in RL_CONFIGS:
    folder = INSTANCE_FOLDER / basename
    files  = sorted(folder.glob('*.txt'))[:amount]
    for f in files:
        ppo_layouts.append(read_instance(str(f), H))

print(f'Total instancias PPO cargadas: {len(ppo_layouts)}')

Total instancias PPO cargadas: 20000


## 3. Cargar modelo pre-entrenado → PPOActorCritic

`PPOActorCritic` es subclase de `CPMPTransformer` con un `value_head` adicional.
Cargamos los pesos SL pre-entrenados con `strict=False` — el `value_head` se inicializa desde cero.

In [6]:
MODEL_NAME = 'v9_dataBSG_250k'

hyperparams = load_hyperparams(MODEL_NAME)
model = PPOActorCritic(**hyperparams).to(device)

# Cargar pesos SL pre-entrenados (strict=False: ignora value_head que no existe en el checkpoint SL)
pretrained = load_model(CPMPTransformer, MODEL_NAME)
missing, unexpected = model.load_state_dict(pretrained.state_dict(), strict=False)
print(f'Pesos SL cargados en PPOActorCritic.')
print(f'  Parámetros totales : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Backbone (policy)  : {sum(p.numel() for n, p in model.named_parameters() if "value_head" not in n):,}')
print(f'  Value head (nuevo) : {sum(p.numel() for p in model.value_head.parameters()):,}')
print(f'  Keys faltantes     : {missing}')   # debe ser solo value_head.*
del pretrained

Pesos SL cargados en PPOActorCritic.
  Parámetros totales : 287,297
  Backbone (policy)  : 285,184
  Value head (nuevo) : 2,113
  Keys faltantes     : ['value_head.0.weight', 'value_head.0.bias', 'value_head.2.weight', 'value_head.2.bias']


## 4. Dataset supervisado (regularización SL)

Mismo dataset BSG que en REINFORCE para evitar catastrofic forgetting.

In [7]:
SL_DATA_FILES = [
    'E4-15-H5_V9_BSG.data',
    'E5-25-H7_V9_BSG.data',
    'E6-45-H10_V9_BSG.data',
    'E7-30-H6_V9_BSG.data',
]

sl_datasets = []
for fname in SL_DATA_FILES:
    path = DATA_FOLDER / fname
    if path.exists():
        sl_datasets.append(H5Dataset(str(path)))
        print(f'  Cargado: {fname}')
    else:
        print(f'  No encontrado (opcional): {fname}')

sl_loader = None
if sl_datasets:
    sl_dataset = ConcatDataset(sl_datasets)
    sl_loader  = DataLoader(
        sl_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=2,
        pin_memory=(device.type == 'cuda'),
        collate_fn=pad_batch_collate,
        drop_last=True,
    )
    print(f'\nDataLoader SL listo: {len(sl_dataset):,} muestras.')
else:
    print('\nSin dataset SL — entrenando solo con PPO (más inestable).')

  Cargado: E4-15-H5_V9_BSG.data
  Cargado: E5-25-H7_V9_BSG.data
  Cargado: E6-45-H10_V9_BSG.data
  Cargado: E7-30-H6_V9_BSG.data

DataLoader SL listo: 249,944 muestras.


## 5. Inicializar PPOTrainer

In [ ]:
trainer = PPOTrainer(
    model=model,
    device=device,
    learning_rate=3e-6,
    # PPO/A2C
    clip_eps=0.2,
    n_ppo_epochs=1,
    mini_batch_size=20000,   # mayor que cualquier rollout → 1 solo grad step por rollout
    value_coeff=0.5,
    # GAE
    gamma=0.99,
    lam=0.95,
    # Sin reward shaping — reward simple para que el crítico aprenda más fácil
    step_reward_coeff=0.0,
    # Entropy
    entropy_coeff=0.01,
    # Regularización SL fuerte y decaimiento lento
    sl_coeff=0.5,
    sl_coeff_min=0.3,
    sl_coeff_decay=0.95,
    # Training
    max_steps=MAX_STEPS,
    grad_clip=1.0,
)
print('PPOTrainer (modo A2C: 1 grad step por rollout) listo.')

## 6. Sanity check

Verifica que el rollout produce trayectorias válidas, que los valores son razonables
y que el ppo_update puede hacer un paso de gradiente sin errores.

In [ ]:
import copy

test_layouts = [copy.deepcopy(ppo_layouts[i]) for i in range(4)]
trajs = trainer.rollout_batch(test_layouts)

print(f'Trayectorias colectadas: {len(trajs)}')
for k, t in enumerate(trajs):
    print(f'  [{k}] T={len(t["actions"])} pasos | resuelto={t["solved"]} | '
          f'reward_total={t["total_reward"]:.1f} | '
          f'adv_mean={float(np.mean(t["advantages"])):.3f}')

stats = trainer.ppo_update(trajs)
print(f'\nA2C update OK:')
print(f'  policy_loss = {stats["policy_loss"]:.4f}')
print(f'  value_loss  = {stats["value_loss"]:.4f}')
print(f'  entropy     = {stats["entropy"]:.4f}')
print(f'  clip%       = {stats["clip_ratio"]*100:.1f}%   ← debe ser ~0% en el primer step')

# Reinicializar con los mismos parámetros
trainer = PPOTrainer(
    model=model, device=device, learning_rate=3e-6,
    clip_eps=0.2, n_ppo_epochs=1, mini_batch_size=20000, value_coeff=0.5,
    gamma=0.99, lam=0.95, step_reward_coeff=0.0, entropy_coeff=0.01,
    sl_coeff=0.5, sl_coeff_min=0.3, sl_coeff_decay=0.95,
    max_steps=MAX_STEPS, grad_clip=1.0,
)
print('\nTrainer reinicializado — listo para el training loop.')

## 7. Training loop PPO

Checkpoint cada `CHECKPOINT_EVERY` epochs en `v9_rl_ppo_checkpoint.pth`.
Para reanudar, ejecuta primero la celda **7b** antes de este loop.

In [10]:
N_EPOCHS          = 50
BATCH_SIZE        = 128      # 4× más que REINFORCE
CHECKPOINT_EVERY  = 5
EVAL_EVERY        = 5
CHECKPOINT_PATH   = str(MODELS_FOLDER / 'v9_rl_ppo_checkpoint.pth')

# Muestra CVS para eval rápida (200 instancias, no vistas en training)
from cpmp.layout import read_file
CVS_FOLDER = Path('instances/benchmarks/CVS')
eval_layouts = []
for dat_file in sorted(CVS_FOLDER.rglob('*.dat'))[:200]:
    try:
        H_file = int(dat_file.parent.name.split('-')[0])
        eval_layouts.append(read_file(str(dat_file), H=H_file + 2))
    except Exception:
        pass
print(f'Instancias CVS para eval rápida: {len(eval_layouts)}')
print(f'Checkpoint: {CHECKPOINT_PATH}')

Instancias CVS para eval rápida: 200
Checkpoint: /mnt/data/Proyectos/Universidad/CPMP-Transformer/models/v9_rl_ppo_checkpoint.pth


In [ ]:
history = []

for epoch in range(1, N_EPOCHS + 1):
    stats = trainer.train_epoch(ppo_layouts, sl_loader=sl_loader, batch_size=BATCH_SIZE)
    history.append(stats)

    print(
        f"Epoch {epoch:3d}/{N_EPOCHS} | "
        f"reward: {stats.get('mean_reward', 0):7.2f} | "
        f"solved: {stats.get('solved_ratio', 0)*100:5.1f}% | "
        f"vf_loss: {stats.get('value_loss', 0):.4f} | "
        f"entropy: {stats.get('entropy', 0):.3f} | "
        f"clip%: {stats.get('clip_ratio', 0)*100:4.1f}% | "
        f"sl_coeff: {trainer.sl_coeff:.3f}"
    )

    if epoch % EVAL_EVERY == 0:
        eval_stats = trainer.evaluate(eval_layouts, max_steps=100)
        print(
            f"  >> Eval CVS (n=200): "
            f"solve_rate={eval_stats['solve_rate']*100:.1f}% | "
            f"mean_steps={eval_stats['mean_steps']:.2f}"
        )

    if epoch % CHECKPOINT_EVERY == 0:
        trainer.save_checkpoint(CHECKPOINT_PATH)

print('\nTraining PPO finalizado.')

  Epoch 1 | batch   5/160 | reward:  -18.07 | solved:  99.7% | ent: 1.488 | clip%: 52.5% | vf_loss: 228.6882
  Epoch 1 | batch  10/160 | reward:  -26.03 | solved:  98.0% | ent: 1.797 | clip%: 59.0% | vf_loss: 428.8162
  Epoch 1 | batch  15/160 | reward:  -16.62 | solved: 100.0% | ent: 1.660 | clip%: 55.3% | vf_loss: 155.0309
  Epoch 1 | batch  20/160 | reward:  -29.96 | solved:  96.6% | ent: 1.666 | clip%: 58.0% | vf_loss: 546.1505
  Epoch 1 | batch  25/160 | reward:  -22.26 | solved:  98.6% | ent: 1.595 | clip%: 55.2% | vf_loss: 336.8437
  Epoch 1 | batch  30/160 | reward:  -33.06 | solved:  95.8% | ent: 1.694 | clip%: 58.8% | vf_loss: 632.4035
  Epoch 1 | batch  35/160 | reward:  -17.40 | solved:  99.8% | ent: 1.627 | clip%: 54.5% | vf_loss: 200.7578
  Epoch 1 | batch  40/160 | reward:  -23.17 | solved:  98.3% | ent: 1.571 | clip%: 54.3% | vf_loss: 365.4439
  Epoch 1 | batch  45/160 | reward:  -28.81 | solved:  96.9% | ent: 1.617 | clip%: 55.5% | vf_loss: 573.2521
  Epoch 1 | batch  

## 7b. Reanudar desde checkpoint

Si interrumpiste el entrenamiento, ejecuta **esta celda** antes del loop de training.
Crea un trainer nuevo con los mismos hiperparámetros y carga el checkpoint.

In [ ]:
# Descomentar para reanudar

# trainer = PPOTrainer(
#     model=model, device=device, learning_rate=1e-5,
#     clip_eps=0.2, n_ppo_epochs=4, mini_batch_size=256, value_coeff=0.5,
#     gamma=0.99, lam=0.95, step_reward_coeff=1.0, entropy_coeff=0.01,
#     sl_coeff=0.3, sl_coeff_min=0.15, sl_coeff_decay=0.85,
#     max_steps=MAX_STEPS, grad_clip=1.0,
# )
# trainer.load_checkpoint(CHECKPOINT_PATH)
# remaining = N_EPOCHS - trainer.epoch
# print(f'Reanudando desde epoch {trainer.epoch} — faltan {remaining} epochs')

## 8. Evaluación final — Benchmark CVS completo (840 instancias)

Compara PPO vs SL base sobre todas las categorías CVS.

In [ ]:
all_cvs     = []
all_cvs_h10 = []

for dat_file in sorted(CVS_FOLDER.rglob('*.dat')):
    try:
        H_file = int(dat_file.parent.name.split('-')[0])
        layout = read_file(str(dat_file), H=H_file + 2)
        if H_file == 10:
            all_cvs_h10.append(layout)
        else:
            all_cvs.append(layout)
    except Exception:
        pass

print(f'CVS (H_archivo < 10): {len(all_cvs)} instancias')
print(f'CVS (H_archivo = 10): {len(all_cvs_h10)} instancias')

# Evaluar PPO
ppo_eval     = trainer.evaluate(all_cvs,     max_steps=100)
ppo_eval_h10 = trainer.evaluate(all_cvs_h10, max_steps=100) if all_cvs_h10 else None

# Evaluar SL base (para comparar)
model_sl = load_model(CPMPTransformer, MODEL_NAME).to(device)
from training.rl_training import RLTrainer
trainer_sl   = RLTrainer(model=model_sl, device=device, max_steps=100)
sl_eval      = trainer_sl.evaluate(all_cvs,     max_steps=100)
sl_eval_h10  = trainer_sl.evaluate(all_cvs_h10, max_steps=100) if all_cvs_h10 else None

print('\n' + '═'*54)
print('  RESULTADOS — H < 10 (instancias relevantes)')
print('═'*54)
print(f"  SL base → solve: {sl_eval['solve_rate']*100:.1f}%  | steps: {sl_eval['mean_steps']:.2f}")
print(f"  PPO RL  → solve: {ppo_eval['solve_rate']*100:.1f}%  | steps: {ppo_eval['mean_steps']:.2f}")
print(f"  BSG ref → solve: 100.0% | steps: 26.90")
print('═'*54)

if ppo_eval_h10:
    print('\n' + '═'*54)
    print('  RESULTADOS — H = 10 (instancias difíciles)')
    print('═'*54)
    print(f"  SL base → solve: {sl_eval_h10['solve_rate']*100:.1f}%  | steps: {sl_eval_h10['mean_steps']:.2f}")
    print(f"  PPO RL  → solve: {ppo_eval_h10['solve_rate']*100:.1f}%  | steps: {ppo_eval_h10['mean_steps']:.2f}")
    print('═'*54)

## 9. Guardar modelo PPO final

In [ ]:
# Guarda el checkpoint final (igual que los intermedios)
trainer.save_checkpoint(CHECKPOINT_PATH)
print(f'Modelo PPO guardado: {CHECKPOINT_PATH}')
print(f'Epoch: {trainer.epoch} | Steps: {trainer.step}')